In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


In [5]:
# Filter fossil fuels only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe colour palette
colours = {
    'Coal': '#0072B2',
    'Oil': '#E69F00',
    'Natural Gas': '#009E73'
}

# Find the region and country with the highest fossil TWh
top_region = df_fossil.groupby('Region')['TWh'].sum().idxmax()
top_country = df_fossil.groupby('Country')['TWh'].sum().idxmax()

# Create treemap
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=colours,
    title=f'Fossil Fuel Dependency: {top_country} and {top_region} Lead in TWh'
)

# Grey out parent nodes (Region and Country)
trace = fig.data[0]
parent_ids = set(trace.parents)

node_colours = [
    '#BDBDBD' if node_id in parent_ids else colours.get(label, '#BDBDBD')
    for node_id, label in zip(trace.ids, trace.labels)
]

fig.update_traces(
    marker=dict(colors=node_colours),
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()

In [6]:

# Filter low-carbon sources
df_low = df.loc[df['Source_Type'] == 'Low-carbon'].copy()

# Aggregate total TWh by country
df_low_country = df_low.groupby('Country', as_index=False)['TWh'].sum()

# Add dummy root node for treemap
df_low_country['All'] = 'Low-carbon'

# Find leading country
top_country = df_low_country.loc[df_low_country['TWh'].idxmax(), 'Country']

# Treemap
fig_tree = px.treemap(
    df_low_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon Energy by Country (Nuclear + Hydro)'
)

fig_tree.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig_tree.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig_tree.show()


# Sort countries by TWh for horizontal bar chart
df_low_sorted = df_low_country.sort_values('TWh', ascending=True)

# Horizontal bar chart
fig_bar = px.bar(
    df_low_sorted,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#0072B2'],
    text='TWh',
    title=f'Leading Low-carbon Energy Producers: {top_country}'
)

fig_bar.update_traces(
    texttemplate='%{text:,.0f} TWh',
    textposition='outside'
)

fig_bar.update_layout(
    xaxis_title='Energy (TWh)',
    yaxis_title='Country',
    showlegend=False,
    margin=dict(t=60, l=10, r=40, b=10)
)

fig_bar.show()


I think the bar chart works better because it's much easier to compare the countries and see which ones produce the most low-carbon energy. The treemap is nice for getting an overall picture of how the energy is distributed, but I find the bar chart easier to read and understand, especially when comparing the actual values.

In [7]:

# Reuse fossil fuel data from Task 1
# Aggregate TWh by Region and Source
df_region = df_fossil.groupby(
    ['Region', 'Source'], as_index=False
)['TWh'].sum()

# Calculate each region's total fossil energy
df_region['Region_Total'] = df_region.groupby(
    'Region'
)['TWh'].transform('sum')

# Calculate percentage share of each fossil source
df_region['Share'] = df_region['TWh'] / df_region['Region_Total']

# Find coal share for each region
coal_shares = df_region[df_region['Source'] == 'Coal'].set_index('Region')['Share']

# Sort regions by coal share (highest first)
region_order = coal_shares.sort_values(ascending=False).index.tolist()

# Add percentage labels to Coal segments only
df_region['Coal_Label'] = np.where(
    df_region['Source'] == 'Coal',
    df_region['Share'].map(lambda x: f'{x:.1%}'),
    ''
)

# Identify the most coal-dependent region
top_region = coal_shares.idxmax()
top_share = coal_shares.max()

# Define colours
colours = {
    'Coal': '#0072B2',
    'Oil': '#BDBDBD',
    'Natural Gas': '#737373'
}

# Create horizontal 100% stacked bar chart
fig = px.bar(
    df_region,
    x='Share',
    y='Region',
    color='Source',
    orientation='h',
    barmode='stack',
    text='Coal_Label',
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map=colours,
    title=f'Most Coal-Dependent Region: {top_region} ({top_share:.1%})'
)

# Format chart
fig.update_traces(
    textposition='inside',
    insidetextanchor='middle'
)

fig.update_layout(
    xaxis_title='Share of Fossil Energy',
    yaxis_title='Region',
    xaxis_tickformat='.0%',
    xaxis_range=[0, 1],
    yaxis=dict(autorange='reversed'),
    legend_title='Energy Source',
    margin=dict(t=70, l=10, r=10, b=10)
)

fig.show()


The treemap makes it easier to see which countries and regions consume the most fossil energy in actual TWh values. The stacked bar chart, however, helps me see which region depends more on coal compared to oil and natural gas. So basically, the treemap shows how much fossil energy is used, while the stacked bar shows how that energy is split.